# AFDM 软件基准：第一步

本 Notebook 只使用 NumPy，在电脑或板卡的 Python 内核中执行相同的软件仿真。
不导入 PYNQ、不加载 Overlay、不读写 ADC/DAC 或 DMA。

操作：按顺序运行全部代码单元格。先看自检是否全为 PASS，再看最后的误码率表。

范围：DAFT/IDAFT、单位化检查、OFDM 退化检查、已知 QPSK 比特的无噪声与 AWGN 收发。
未包括 CPP、同步、多径、多普勒、导频、信道估计和射频收发。不能把本 Notebook 的结果当作板上 AFDM 已通过。

约定来自 [AFDM 原论文](https://arxiv.org/html/2204.12798v2) §II-B、§III-A/C：
`A = Lambda(c2) @ F @ Lambda(c1)`，`Lambda(c)[n,n] = exp(-j*2*pi*c*n*n)`。
发送使用 `A.H`，接收使用 `A`；FFT 与 IFFT 都使用 `norm="ortho"`。
此处 chirp 参数只用于基准自检，尚未按多普勒信道设计。


In [ ]:
import numpy as np
import math

SEED = 20261117
N = 64
BLOCKS = 512
c1 = 1.0 / (2 * N)
c2 = np.sqrt(2.0) / (N * N)
SNR_DB = np.arange(0, 11, 2)

def idaft(symbols, c1, c2):
    """输入最后一维为 DAFT 符号，输出单位化的时域波形。"""
    symbols = np.asarray(symbols, dtype=np.complex128)
    n = np.arange(symbols.shape[-1])
    return np.exp(2j*np.pi*c1*n*n) * np.fft.ifft(
        symbols * np.exp(2j*np.pi*c2*n*n), axis=-1, norm='ortho')

def daft(samples, c1, c2):
    """输入时域采样，输出 DAFT 域符号。"""
    samples = np.asarray(samples, dtype=np.complex128)
    n = np.arange(samples.shape[-1])
    return np.exp(-2j*np.pi*c2*n*n) * np.fft.fft(
        samples * np.exp(-2j*np.pi*c1*n*n), axis=-1, norm='ortho')

def qpsk_map(bits):
    bits = np.asarray(bits)
    return ((1.0-2.0*bits[..., 0]) + 1j*(1.0-2.0*bits[..., 1])) / np.sqrt(2.0)

def qpsk_bits(symbols):
    return np.stack([symbols.real < 0, symbols.imag < 0], axis=-1).astype(np.uint8)

print('NumPy:', np.__version__)
print('N =', N, ', blocks =', BLOCKS, ', seed =', SEED)
print('c1 =', c1, ', c2 =', c2)


In [ ]:
rng = np.random.default_rng(SEED)
z = (rng.standard_normal((4, N)) + 1j*rng.standard_normal((4, N))) / np.sqrt(2)
wave = idaft(z, c1, c2)
restored = daft(wave, c1, c2)
roundtrip_error = float(np.max(np.abs(restored-z)))
assert roundtrip_error < 1e-12
print('PASS 1: AFDM 自恢复，最大误差 =', roundtrip_error)

# 独立构造完整 DAFT 矩阵，检查 FFT 实现的符号、索引及归一化。
m = np.arange(N)[:, None]
n = np.arange(N)[None, :]
A = np.exp(-2j*np.pi*(c2*m*m + m*n/N + c1*n*n)) / np.sqrt(N)
matrix_error = float(np.max(np.abs(restored - wave @ A.T)))
inverse_matrix_error = float(np.max(np.abs(wave - z @ A.conj())))
assert matrix_error < 2e-12 and inverse_matrix_error < 2e-12
print('PASS 2: DAFT/IDAFT 与完整矩阵公式一致')

energy_error = float(np.max(np.abs(
    np.sum(np.abs(wave)**2, axis=-1) - np.sum(np.abs(z)**2, axis=-1))))
assert energy_error < 1e-10
print('PASS 3: 变换前后能量守恒')

ofdm_reference = np.fft.ifft(z, axis=-1, norm='ortho')
ofdm_error = float(np.max(np.abs(idaft(z, 0, 0)-ofdm_reference)))
assert ofdm_error < 1e-12
assert np.max(np.abs(daft(wave, 0, 0)-np.fft.fft(wave, axis=-1, norm='ortho'))) < 1e-12
print('PASS 4: c1=c2=0 时退化为单位化 OFDM，误差 =', ofdm_error)

tx_bits = rng.integers(0, 2, size=(BLOCKS, N, 2), dtype=np.uint8)
symbols = qpsk_map(tx_bits)
tx_afdm = idaft(symbols, c1, c2)
tx_ofdm = idaft(symbols, 0, 0)
clean_afdm_errors = int(np.count_nonzero(qpsk_bits(daft(tx_afdm, c1, c2)) != tx_bits))
clean_ofdm_errors = int(np.count_nonzero(qpsk_bits(daft(tx_ofdm, 0, 0)) != tx_bits))
assert clean_afdm_errors == 0 and clean_ofdm_errors == 0
print('PASS 5: 无噪声已知比特收发：AFDM/OFDM 均为 0 /', tx_bits.size, '个错误')


## 加入白噪声：已知比特的误码统计
AWGN 无时延与多普勒，这一段无需前缀。它是调制/解调软件检查，不是完整无线信道基准。

In [ ]:
# SNR 定义：Es/N0（每个复数符号的信噪比），不是 Eb/N0。
# 单位化 QPSK 的 Es=1；复噪声方差为 1/SNR，两实分量各为其一半。
# 两种波形使用相同的比特数、块长、平均功率和时域噪声样本。
rows = []
print('SNR(dB)  AFDM错误/比特数   OFDM错误/比特数   AFDM_BER   OFDM_BER   理论QPSK_BER')
for snr_db in SNR_DB:
    snr_linear = 10.0**(float(snr_db)/10.0)
    noise = (rng.standard_normal(tx_afdm.shape) + 1j*rng.standard_normal(tx_afdm.shape)) / np.sqrt(2*snr_linear)
    afdm_errors = int(np.count_nonzero(qpsk_bits(daft(tx_afdm+noise, c1, c2)) != tx_bits))
    ofdm_errors = int(np.count_nonzero(qpsk_bits(daft(tx_ofdm+noise, 0, 0)) != tx_bits))
    afdm_ber, ofdm_ber = afdm_errors/tx_bits.size, ofdm_errors/tx_bits.size
    theory = 0.5*math.erfc(math.sqrt(snr_linear/2.0))
    tolerance = max(5*math.sqrt(theory*(1-theory)/tx_bits.size), 5/tx_bits.size)
    assert abs(afdm_ber-theory) < tolerance
    assert abs(ofdm_ber-theory) < tolerance
    row = dict(snr_db=int(snr_db), bits=int(tx_bits.size), afdm_errors=afdm_errors,
               ofdm_errors=ofdm_errors, afdm_ber=afdm_ber, ofdm_ber=ofdm_ber,
               qpsk_awgn_theory=theory)
    rows.append(row)
    print('%7d  %5d/%-8d  %5d/%-8d  %.6f   %.6f   %.6f' % (
        snr_db, afdm_errors, tx_bits.size, ofdm_errors, tx_bits.size,
        afdm_ber, ofdm_ber, theory))
print('PASS 6: 两种波形的 AWGN 误码率均在理论值的统计容差内')
print('当前结果：软件基准通过。AWGN 下不期待 AFDM 获得额外优势。')


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import uuid

result_dir = Path.cwd() / ('afdm_results_' + uuid.uuid4().hex[:8])
result_dir.mkdir(exist_ok=False)
summary = dict(scope='software-only; no RF, CPP, multipath or Doppler',
               host_clock=datetime.now().isoformat(), seed=SEED, n=N, blocks=BLOCKS,
               numpy_version=np.__version__, c1=c1, c2=float(c2), snr_definition='Es/N0',
               roundtrip_max_error=roundtrip_error, matrix_max_error=matrix_error,
               inverse_matrix_max_error=inverse_matrix_error, energy_error=energy_error,
               ofdm_degenerate_error=ofdm_error, clean_afdm_bit_errors=clean_afdm_errors,
               clean_ofdm_bit_errors=clean_ofdm_errors, awgn=rows)
(result_dir/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
np.savez(result_dir/'reference_vectors.npz', bits=tx_bits[0], symbols=symbols[0],
         tx_afdm=tx_afdm[0], tx_ofdm=tx_ofdm[0], c1=c1, c2=c2, seed=SEED)
print('结果保存位置：', result_dir)
print('下一步：CPP 与静态多径；之后才加入多普勒、信道估计和射频回放。')
